In [1]:
from pyspark.sql import SparkSession

TEAM = "team14"

spark = (
    SparkSession.builder
    .appName(f"{TEAM}-stage3-cancelled-dev")
    .master("yarn")
    # Hive metastore URI is a Hadoop conf, so use spark.hadoop.*
    .config("spark.hadoop.hive.metastore.uris", "thrift://hadoop-02.uni.innopolis.ru:9883")
    .config("spark.sql.warehouse.dir", f"hdfs:///user/{TEAM}/project/hive/warehouse")
    # Yarn resources (requested from cluster scheduler; tweak if queues reject allocations)
    .config("spark.executor.instances", "4")      # sum can exceed UI "max allocation" limits
    .config("spark.executor.cores", "3")           # requested per executor
    .config("spark.executor.memory", "4g")         # stays under typical per-container ceilings
    .config("spark.executor.memoryOverhead", "1536m")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.memoryOverhead", "1024m")
    .config("spark.dynamicAllocation.enabled", "false")  # keep allocation predictable during CV/grid
    .config("spark.sql.shuffle.partitions", "512")
    # CrossValidator parallelism>1 triggers multiprocessing pools on driver; stick to 1 to reduce crashes
    .config("spark.default.parallelism", "128")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/02 00:44:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
26/05/02 00:44:54 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
26/05/02 00:44:54 WARN Utils: Serv

In [2]:
from pyspark.sql import functions as F

DB = "team14_projectdb"
TABLE = f"{DB}.flights_part"
SEED = 42

In [3]:
df = spark.table(TABLE)

base = (
    df
    .select(
        "airline_code", "origin", "dest",
        "fl_number", "distance",
        "fl_date", "crs_dep_time", "crs_arr_time",
        "cancelled"
    )
    .withColumn("label", F.col("cancelled").cast("int"))
    .drop("cancelled")
)

# time features from fl_date (epoch ms)
base = (
    base
    .withColumn("flight_ts", F.from_unixtime((F.col("fl_date")/1000).cast("long")).cast("timestamp"))
    .withColumn("month", F.month("flight_ts"))
    .withColumn("day_of_week", F.dayofweek("flight_ts"))  # 1..7
    .drop("flight_ts")
)

# numeric cast
base = base.withColumn("distance_km", F.col("distance").cast("double")).drop("distance")

# drop a few null rows (у тебя это буквально 14 строк)
clean = base.filter(
    F.col("crs_dep_time").isNotNull() &
    F.col("crs_arr_time").isNotNull() &
    F.col("distance_km").isNotNull()
)

clean.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|    0|2920847|
|    1|  79139|
+-----+-------+



In [12]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Separate feature matrices:
# - OHE (+ numeric) for LinearSVC
# - OHE ONLY (no numeric cols) for NaiveBayes Bernoulli — NB requires features in {0, 1}
# - Index-only vectors for RandomForest (see RF cell; OHE blows up cardinality for origin/dest)

cat_cols = ["airline_code", "origin", "dest"]
num_cols = ["fl_number", "distance_km", "month", "day_of_week", "crs_dep_time", "crs_arr_time"]

indexers_ohe = [StringIndexer(inputCol=c, outputCol=f"{c}_idx_ohe", handleInvalid="keep") for c in cat_cols]
encoder = OneHotEncoder(
    inputCols=[f"{c}_idx_ohe" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
)
assembler_ohe = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in cat_cols] + num_cols,
    outputCol="features",
    handleInvalid="keep",
)

prep_ohe = Pipeline(stages=indexers_ohe + [encoder, assembler_ohe])
data_ohe = prep_ohe.fit(clean).transform(clean).select("features", "label")

train_ohe, test_ohe = data_ohe.randomSplit([0.7, 0.3], seed=SEED)
train_ohe = train_ohe.cache()
test_ohe = test_ohe.cache()

print(
    "OHE(+numeric) split — train:",
    train_ohe.count(),
    "test:",
    test_ohe.count(),
)

# --- NaiveBayes-only features: categorical OHE only (values are 0/1) ---
assembler_nb = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep",
)

prep_nb = Pipeline(stages=indexers_ohe + [encoder, assembler_nb])
data_nb = prep_nb.fit(clean).transform(clean).select("features", "label")

train_nb, test_nb = data_nb.randomSplit([0.7, 0.3], seed=SEED)
train_nb = train_nb.cache()
test_nb = test_nb.cache()

print(
    "NB(OHE-only) split — train:",
    train_nb.count(),
    "test:",
    test_nb.count(),
)

OHE(+numeric) split — train: 2100860 test: 899126


NB(OHE-only) split — train: 2100860 test: 899126


In [5]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

e_auc_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
e_auc_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

e_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

In [6]:
def eval_on_test(model, test_df, name: str):
    pred = model.transform(test_df).cache()
    out = {
        "model": name,
        "auc_roc": float(e_auc_roc.evaluate(pred)),
        "auc_pr": float(e_auc_pr.evaluate(pred)),
        "f1": float(e_f1.evaluate(pred)),
    }
    pred.unpersist()
    return out

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# RandomForest tip:
# - DO NOT OneHotEncode high-cardinality categoricals (origin/dest): huge + unstable on resource limits.
#
# IMPORTANT: Spark trees treat indexed categoricals specially: maxBins must be >= cardinality of EACH categorical split.

rf_cat_cols = ["airline_code", "origin", "dest"]
rf_num_cols = ["fl_number", "distance_km", "month", "day_of_week", "crs_dep_time", "crs_arr_time"]

rf_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx_rf", handleInvalid="keep")
    for c in rf_cat_cols
]

rf_assembler = VectorAssembler(
    inputCols=[f"{c}_idx_rf" for c in rf_cat_cols] + rf_num_cols,
    outputCol="features",
    handleInvalid="keep",
)

rf_prep_pipe = Pipeline(stages=rf_indexers + [rf_assembler])

# Fit indexer stage once so we know label cardinalities (+1 for unseen bucket via handleInvalid='keep').
rf_prep_model = rf_prep_pipe.fit(clean)

def _si_num_labels(si_model) -> int:
    """PySpark naming differs across versions: prefer labels length."""
    lbls = getattr(si_model, "labels", None)
    if lbls is None:
        lbls_arr = getattr(si_model, "labelsArray", None)
        # labelsArray exists on some pipelines as list[str]
        if lbls_arr is not None:
            return int(len(lbls_arr))
        raise AttributeError("StringIndexerModel has neither .labels nor .labelsArray")
    return int(len(lbls))

rf_max_cardinality = max(
    _si_num_labels(m) + 1 for m in rf_prep_model.stages[:-1]
)
bins_min = rf_max_cardinality

print("RF indexer maxCardinality (approx with unseen bucket):", rf_max_cardinality)

rf_data = rf_prep_model.transform(clean).select("features", "label")
rf_train, rf_test = rf_data.randomSplit([0.7, 0.3], seed=SEED)
rf_train = rf_train.cache()
rf_test = rf_test.cache()
print(
    "RF split — train:",
    rf_train.count(),
    "test:",
    rf_test.count(),
)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=SEED)

# 27 combos — maxBins MUST stay >= categorical cardinality requirement.
bins_lo = bins_min
bins_mid = bins_min + 128
bins_hi = bins_min + 256

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [6, 10, 14])
    .addGrid(rf.maxBins, [bins_lo, bins_mid, bins_hi])
    .addGrid(rf.subsamplingRate, [0.5, 0.75, 1.0])
    .build()
)

rf_cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_grid,
    evaluator=e_auc_pr,
    numFolds=3,
    parallelism=1,
)

rf_cv_model = rf_cv.fit(rf_train)
rf_best = rf_cv_model.bestModel

rf_metrics = eval_on_test(rf_best, rf_test, "RandomForest(best)")
rf_metrics

RF indexer maxCardinality (approx with unseen bucket): 381


RF split — train: 2100860 test: 899126


26/05/02 00:52:23 WARN DAGScheduler: Broadcasting large task binary with size 1483.5 KiB
26/05/02 00:52:26 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/05/02 00:52:29 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
26/05/02 00:52:34 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB
26/05/02 00:52:39 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/05/02 00:52:54 WARN DAGScheduler: Broadcasting large task binary with size 1487.4 KiB
26/05/02 00:52:57 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/05/02 00:53:01 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/05/02 00:53:06 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/05/02 00:53:12 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/05/02 00:53:27 WARN DAGScheduler: Broadcasting large task binary with size 1422.9 KiB
26/05/02 00:53:30 WARN DAGScheduler: Broadcas

{'model': 'RandomForest(best)',
 'auc_roc': 0.7255050960945607,
 'auc_pr': 0.0782469089398294,
 'f1': 0.9609810012544325}

In [10]:
from pyspark.ml.classification import LinearSVC
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# LinearSVC = SVM in Spark (do NOT tune maxIter per course rules).
svm = LinearSVC(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    standardization=True,
)

# 3 hp × 3 values = 27. Aggregator depth is solver-related (algorithm-ish) vs reg/threshold tuning.
svm_grid = (
    ParamGridBuilder()
    .addGrid(svm.regParam, [0.01, 0.05, 0.2])
    .addGrid(svm.threshold, [0.35, 0.5, 0.65])
    .addGrid(svm.aggregationDepth, [2, 3, 4])
    .build()
)

svm_cv = CrossValidator(
    estimator=svm,
    estimatorParamMaps=svm_grid,
    evaluator=e_auc_pr,  # report PR on test separately; aligns with imbalance
    numFolds=3,
    parallelism=1,
)

svm_cv_model = svm_cv.fit(train_ohe)
svm_best = svm_cv_model.bestModel

svm_metrics = eval_on_test(svm_best, test_ohe, "LinearSVC(best)")
svm_metrics

26/05/02 01:53:31 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


{'model': 'LinearSVC(best)',
 'auc_roc': 0.5740198788765554,
 'auc_pr': 0.033562483475831434,
 'f1': 0.9609699550337863}

In [13]:
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Bernoulli NB requires ONLY binary {0,1} features — use train_nb built from OHE categoricals only (no numeric cols).
# Multinomial NB can use nonnegative count-like features; OHE 0/1 counts work too.

nb = NaiveBayes(featuresCol="features", labelCol="label", predictionCol="prediction")

# 27 combos (3×3×3): smoothing × thresholds × modelType
nb_grid = (
    ParamGridBuilder()
    .addGrid(nb.smoothing, [1e-4, 5e-4, 2e-3])
    .addGrid(nb.thresholds, [[0.30, 0.70], [0.40, 0.60], [0.45, 0.55]])
    .addGrid(nb.modelType, ["bernoulli", "multinomial", "multinomial"])
    .build()
)

nb_cv = CrossValidator(
    estimator=nb,
    estimatorParamMaps=nb_grid,
    evaluator=e_auc_pr,
    numFolds=3,
    parallelism=1,
)

nb_cv_model = nb_cv.fit(train_nb)
nb_best = nb_cv_model.bestModel

nb_metrics = eval_on_test(nb_best, test_nb, "NaiveBayes(best)")
nb_metrics

{'model': 'NaiveBayes(best)',
 'auc_roc': 0.5209109054346488,
 'auc_pr': 0.02937063226521807,
 'f1': 0.9605218901737237}

In [14]:
results = [rf_metrics, svm_metrics, nb_metrics]
res_df = spark.createDataFrame(results)
res_df.orderBy(F.desc("auc_pr")).show(truncate=False)

+--------------------+------------------+------------------+------------------+
|auc_pr              |auc_roc           |f1                |model             |
+--------------------+------------------+------------------+------------------+
|0.0782469089398294  |0.7255050960945607|0.9609810012544325|RandomForest(best)|
|0.033562483475831434|0.5740198788765554|0.9609699550337863|LinearSVC(best)   |
|0.02937063226521807 |0.5209109054346488|0.9605218901737237|NaiveBayes(best)  |
+--------------------+------------------+------------------+------------------+



In [ ]:
# --- Save models + predictions + evaluation (Stage III deliverables) ---
# Paths under your HDFS home: hdfs:///user/team14/project/...
import subprocess
import os

BASE = f"hdfs:///user/{TEAM}/project"
MODEL_OUT = {
    "rf": f"{BASE}/models/model_rf",
    "svm": f"{BASE}/models/model_svm",
    "nb": f"{BASE}/models/model_nb",
}
PRED_OUT = {
    "rf": f"{BASE}/output/model_rf_predictions",
    "svm": f"{BASE}/output/model_svm_predictions",
    "nb": f"{BASE}/output/model_nb_predictions",
}
EVAL_OUT = f"{BASE}/output/evaluation.csv"
TRAIN_JSON = f"{BASE}/data/train"
TEST_JSON = f"{BASE}/data/test"

os.makedirs("output", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Refit prep pipelines (same data/seed as before → same transforms)
prep_ohe_model = prep_ohe.fit(clean)
prep_nb_model = prep_nb.fit(clean)

from pyspark.ml import PipelineModel

# Full scoring pipelines: feature prep + best classifier
rf_full = PipelineModel(stages=[rf_prep_model, rf_best])
svm_full = PipelineModel(stages=[prep_ohe_model, svm_best])
nb_full = PipelineModel(stages=[prep_nb_model, nb_best])

rf_full.write().overwrite().save(MODEL_OUT["rf"])
svm_full.write().overwrite().save(MODEL_OUT["svm"])
nb_full.write().overwrite().save(MODEL_OUT["nb"])
print("Saved models:", MODEL_OUT)

# Predictions (label + prediction only, 1 partition) — use same test sets as eval_on_test
pred_rf = rf_best.transform(rf_test).select("label", "prediction")
pred_svm = svm_best.transform(test_ohe).select("label", "prediction")
pred_nb = nb_best.transform(test_nb).select("label", "prediction")

for name, df_pred, path in [
    ("rf", pred_rf, PRED_OUT["rf"]),
    ("svm", pred_svm, PRED_OUT["svm"]),
    ("nb", pred_nb, PRED_OUT["nb"]),
]:
    df_pred.coalesce(1).write.mode("overwrite").format("csv").option("header", True).save(path)
    print("Saved predictions:", name, "->", path)

# Evaluation summary table
rows = [
    (rf_metrics["model"], rf_metrics["auc_roc"], rf_metrics["auc_pr"], rf_metrics["f1"]),
    (svm_metrics["model"], svm_metrics["auc_roc"], svm_metrics["auc_pr"], svm_metrics["f1"]),
    (nb_metrics["model"], nb_metrics["auc_roc"], nb_metrics["auc_pr"], nb_metrics["f1"]),
]
eval_df = spark.createDataFrame(rows, ["model", "auc_roc", "auc_pr", "f1"])
eval_df.coalesce(1).write.mode("overwrite").format("csv").option("header", True).save(EVAL_OUT)
print("Saved evaluation ->", EVAL_OUT)

# Stage III checklist: persist train/test feature matrices as JSON (RF split matches your 70/30)
rf_train.write.mode("overwrite").format("json").save(TRAIN_JSON)
rf_test.write.mode("overwrite").format("json").save(TEST_JSON)
print("Saved train/test JSON ->", TRAIN_JSON, TEST_JSON)


def _hdfs_getmerge(hdfs_dir: str, local_file: str) -> None:
    subprocess.run(f'hdfs dfs -getmerge {hdfs_dir} {local_file}', shell=True, check=True)


_hdfs_getmerge(PRED_OUT["rf"], "output/model_rf_predictions.csv")
_hdfs_getmerge(PRED_OUT["svm"], "output/model_svm_predictions.csv")
_hdfs_getmerge(PRED_OUT["nb"], "output/model_nb_predictions.csv")
_hdfs_getmerge(EVAL_OUT, "output/evaluation.csv")
print("Pulled CSVs into output/")